# RMSProp Simulations - Non-Convex

## Parameters

In [0]:
from Solver import RMSPropMomentum, NonlocalSolverMomentumRMSProp
from sklearn.model_selection import ParameterGrid
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os
import jax 
import jax.numpy as jnp

param_grid = {'lr': [0.1, 0.01], 'beta': [0.0, 0.9, 0.99]}
n_learning_rates = len(param_grid['lr'])
param_list = list(ParameterGrid(param_grid))

dL = lambda y: y * (y**2 - 1)
f = lambda x, y: 0.0

# Crear carpeta para guardar figuras si no existe
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

## RMSProp - Discrete

In [0]:
# --- RMSProp (con momentum) | PNGs separados por condición inicial ---

inits = [0.1, -0.1]

for theta_initial in inits:
    # Crear subplots para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Iterar por cada learning rate
    for i, lr in enumerate(param_grid['lr']):

        # Filtrar parámetros por este LR
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs según LR (manteniendo tus valores)
        if lr == 0.1:
            epochs = 15
        elif lr == 0.01:
            epochs = 100
            
        # Simulaciones RMSProp
        for params in filtered_params:
            print(f'\nRMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver = RMSPropMomentum(dL=dL, lr=lr, beta=params['beta'], epochs=epochs)
            solver.solve(theta_initial=theta_initial)

            label = f"beta={params['beta']}"

            # Theta_k
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v_k (gradiente al cuadrado acumulado / EMA)
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Layouts para ESTA condición inicial
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for the RMSProp Optimizer — θ₀={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="k")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta_k")

    fig_v.update_layout(
        title_text=f'Squared gradients convergence trajectories for the RMSProp Optimizer — θ₀={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_v.update_xaxes(title_text="k")
    fig_v.update_yaxes(title_text="v_k")

    # Sufijo de archivo (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"rmsprop_theta_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"rmsprop_v_ncvx_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")



## Nonlocal RMSProp

In [0]:
# --- Nonlocal Continuous RMSProp | PNGs separados por condición inicial ---

inits = [0.1, -0.1]

for theta_initial in inits:
    # 1) Crear subplots para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Títulos
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for the first-order nonlocal continuous RMSProp — θ₀={theta_initial}'
    )
    fig_v.update_layout(
        title_text=f'v over time for the first-order nonlocal continuous RMSProp — θ₀={theta_initial}'
    )

    # 2) Barrer learning rates
    for i, lr in enumerate(param_grid['lr']):
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs según LR (manteniendo tus valores)
        if lr == 0.1:
            epochs = 15
        elif lr == 0.01:
            epochs = 100

        t = [1e-12, epochs * lr]

        # 3) Simulaciones por configuración
        for params in filtered_params:
            print(f'\nNonlocal Continuous RMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver = NonlocalSolverMomentumRMSProp(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], beta=params['beta']
            )
            t_values, y_values = solver.solve()

            label = f"beta={params['beta']}"

            # Theta(t)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v(t) guardado por el solver
            denominators = np.asarray(solver._last_v)  # columnas: [t, v]
            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # 4) Ejes y tamaños (para ESTA θ0)
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta(t)")
    fig_theta.update_layout(width=1500, height=600)

    fig_v.update_xaxes(title_text="t/alpha")
    fig_v.update_yaxes(title_text="v(t)")
    fig_v.update_layout(width=1500, height=600)

    # 5) Guardado con sufijo por condición inicial (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    fig_theta.write_image(os.path.join(figures_dir, f"nonlocal_rmsprop_theta_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"nonlocal_rmsprop_v_ncvx_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")


## Both Models Together

In [0]:
# --- RMSProp vs. Nonlocal Continuous RMSProp
# --- PNGs separados por condición inicial θ0 ---

config_colors = {
    (0.0): 'blue',
    (0.9): 'green',
    (0.99): 'red'
}

inits = [0.1, -0.1]

for theta_initial in inits:
    # Crear figuras para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1,
        cols=2,  # una columna por LR
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1,
        cols=2,  # una columna por LR
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Barrer learning rates
    for i, lr in enumerate(param_grid['lr']):

        # Filtrar parámetros por este LR
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs según LR (manteniendo tus valores)
        if lr == 0.1:
            epochs = 15
        elif lr == 0.01:
            epochs = 100

        t = [1e-12, epochs * lr]

        # -------- RMSProp (discreto) --------
        for params in filtered_params:
            color = config_colors[params['beta']]
            print(f'\nRMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver = RMSPropMomentum(dL=dL, lr=lr, beta=params['beta'], epochs=epochs)
            solver.solve(theta_initial=theta_initial)

            # θ_k
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.05), color=color),
                name=f'RMSProp beta={params["beta"]}',
                legendgroup=f'RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v_k
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.05), color=color),
                name=f'RMSProp beta={params["beta"]}',
                legendgroup=f'RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

        # ---- RMSProp continuo no local ----
        for params in filtered_params:
            color = config_colors[params['beta']]
            print(f'\nNonlocal Continuous RMSProp Configuration: {params}, theta_initial={theta_initial}')

            solver_nonlocal = NonlocalSolverMomentumRMSProp(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], beta=params['beta']
            )
            t_values, y_values = solver_nonlocal.solve()

            # θ(t)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal RMSProp beta={params["beta"]}',
                legendgroup=f'Nonlocal RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v(t)
            denominators = np.asarray(solver_nonlocal._last_v)  # columnas [t, v]
            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal RMSProp beta={params["beta"]}',
                legendgroup=f'Nonlocal RMSProp beta={params["beta"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Layouts para ESTA condición inicial
    fig_theta.update_layout(
        title_text=f'Theta values convergence trajectories for Nonlocal Continuous RMSProp — θ₀={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta values")

    fig_v.update_layout(
        title_text=f'Squared gradients (v) trajectories for Nonlocal Continuous RMSProp — θ₀={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_v.update_xaxes(title_text="t/alpha")
    fig_v.update_yaxes(title_text="v values")

    # Sufijo de archivo (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"rmsprop_vs_nonlocal_theta_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"rmsprop_vs_nonlocal_v_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")
